In [ ]:
import numpy as np
import vtk
from vtk.util import numpy_support
from mrvis.utils.algorithms import convective_acceleration

# Generate a synthetic vector field of the singular field line from Hesse et al.
size = 64
xmin, xmax = -3, 3
ymin, ymax = -1, 4
zmin, zmax = -3, 3

linx = np.linspace(xmin, xmax, size)
liny = np.linspace(ymin, ymax, size)
linz = np.linspace(zmin, zmax, size)

origin   = (linx[0], liny[0], linz[0])
spacing  = (linx[1]-linx[0], liny[1]-liny[0], linz[1]-linz[0])

x, y, z = np.meshgrid(linx, liny, linz, indexing="ij")

nx, ny, nz = x.shape 

field = np.array([
    (y - 2) ** 2 - 1 + z ** 2,   #  Fx
    -x,                          #  Fy
    np.ones_like(x)*0.1           #  Fz  (constant 1.0)
])

# Compute the convective acceleration of the field
accel = convective_acceleration(field, linx, liny, linz)

# Create a VTK image data object
img = vtk.vtkImageData()
img.SetDimensions(nx, ny, nz)
img.SetOrigin(origin)
img.SetSpacing(spacing)              

# reshape field to (nPoints, 3) and convert to VTK array with fortran order
vtk_vec = numpy_support.numpy_to_vtk(
    num_array=field.reshape(3, -1, order="F").T.astype(np.float32), 
    deep=True,
    array_type=vtk.VTK_FLOAT
)
vtk_vec.SetName("Vectors")      # visible name in ParaView

accel_flat = accel.reshape(3, -1, order="F").T.astype(np.float32)  

vtk_accel = numpy_support.numpy_to_vtk(accel_flat, deep=True)
vtk_accel.SetNumberOfComponents(3)
vtk_accel.SetName("Convective Acceleration")

img.GetPointData().SetVectors(vtk_vec)
img.GetPointData().AddArray(vtk_accel)  

writer = vtk.vtkXMLImageDataWriter()
writer.SetFileName("../data/singular_field_line.vti")
writer.SetInputData(img)
writer.Write()

1